In [12]:
import os
import shutil
import json
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Locate SFT model.safetensors
print("Locating SFT model.safetensors...")
st_candidates = list(Path("/kaggle/input").rglob("model.safetensors")) + list(Path("/kaggle/working").rglob("model.safetensors"))
st_candidates = [f for f in st_candidates if "sft" in str(f).lower() or "checkpoints" in str(f).lower()]
if not st_candidates:
    st_candidates = list(Path("/kaggle/input").rglob("model.safetensors"))

if not st_candidates:
    raise FileNotFoundError("Could not locate model.safetensors in /kaggle/input or /kaggle/working.")

src_dir = st_candidates[0].parent
print(f"✓ Found SFT weights at: {src_dir}")

# 2. Stage model into /kaggle/working/sft_model
model_dir = Path("/kaggle/working/sft_model")
model_dir.mkdir(parents=True, exist_ok=True)

for f in src_dir.glob("*"):
    if f.is_file() and not (model_dir / f.name).exists():
        shutil.copy2(f, model_dir / f.name)

# Ensure tokenizer files are copied
tok_candidates = list(Path("/kaggle/input").rglob("tokenizer.json"))
if tok_candidates:
    for tok_f in tok_candidates[0].parent.glob("tokenizer*"):
        if not (model_dir / tok_f.name).exists():
            shutil.copy2(tok_f, model_dir / tok_f.name)

# 3. Inline configuration_qaptaan.py
config_code = '''"""QaptaanLM-0.75B Configuration."""
from transformers.configuration_utils import PretrainedConfig

class QaptaanConfig(PretrainedConfig):
    model_type = "qaptaan"
    keys_to_ignore_at_inference = ["past_key_values"]

    def __init__(
        self,
        vocab_size: int = 248320,
        hidden_size: int = 1024,
        intermediate_size: int = 3584,
        num_hidden_layers: int = 24,
        num_attention_heads: int = 8,
        num_key_value_heads: int = 2,
        head_dim: int = 256,
        rms_norm_eps: float = 1e-6,
        tie_word_embeddings: bool = True,
        max_position_embeddings: int = 262144,
        rope_theta: float = 10000000.0,
        partial_rotary_factor: float = 0.25,
        attn_output_gate: bool = True,
        full_attention_interval: int = 4,
        linear_key_head_dim: int = 128,
        linear_value_head_dim: int = 128,
        linear_num_key_heads: int = 16,
        linear_num_value_heads: int = 16,
        linear_conv_kernel_dim: int = 4,
        hidden_act: str = "silu",
        initializer_range: float = 0.02,
        use_cache: bool = True,
        bos_token_id: int = None,
        eos_token_id: int = 248044,
        pad_token_id: int = 248044,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = head_dim
        self.rms_norm_eps = rms_norm_eps
        self.tie_word_embeddings = tie_word_embeddings
        self.max_position_embeddings = max_position_embeddings
        self.rope_theta = rope_theta
        self.partial_rotary_factor = partial_rotary_factor
        self.attn_output_gate = attn_output_gate
        self.full_attention_interval = full_attention_interval
        self.linear_key_head_dim = linear_key_head_dim
        self.linear_value_head_dim = linear_value_head_dim
        self.linear_num_key_heads = linear_num_key_heads
        self.linear_num_value_heads = linear_num_value_heads
        self.linear_conv_kernel_dim = linear_conv_kernel_dim
        self.hidden_act = hidden_act
        self.initializer_range = initializer_range
        self.use_cache = use_cache

        self.layer_types = []
        for i in range(num_hidden_layers):
            if (i + 1) % full_attention_interval == 0:
                self.layer_types.append("full_attention")
            else:
                self.layer_types.append("linear_attention")

        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
'''
with open(model_dir / "configuration_qaptaan.py", "w", encoding="utf-8") as f:
    f.write(config_code)

# 4. Inline modeling_qaptaan.py
modeling_code = '''"""QaptaanLM-0.75B PyTorch Model Implementation."""
import math
from typing import Any, Dict, List, Optional, Tuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutputWithPast, BaseModelOutputWithPast
from transformers.modeling_utils import PreTrainedModel
from transformers.generation import GenerationMixin
from .configuration_qaptaan import QaptaanConfig

class QaptaanCache:
    def __init__(self):
        self.conv_states: Dict[int, torch.Tensor] = {}
        self.recurrent_states: Dict[int, torch.Tensor] = {}
        self.key_cache: Dict[int, torch.Tensor] = {}
        self.value_cache: Dict[int, torch.Tensor] = {}
        self._seen_tokens: int = 0

    def get_seq_length(self, layer_idx: Optional[int] = 0) -> int:
        return self._seen_tokens

    def update(self, key_states: torch.Tensor, value_states: torch.Tensor, layer_idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if layer_idx not in self.key_cache:
            self.key_cache[layer_idx] = key_states
            self.value_cache[layer_idx] = value_states
        else:
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_states], dim=2)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_states], dim=2)
        return self.key_cache[layer_idx], self.value_cache[layer_idx]

class QaptaanRMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class QaptaanRMSNormGated(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor, gate: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        normed = x * torch.rsqrt(variance + self.eps) * self.weight
        return normed * F.silu(gate)

class QaptaanRotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 262144, base: float = 10000000.0):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2, dtype=torch.float32) / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(query: torch.Tensor, key: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, rotary_dim: int) -> Tuple[torch.Tensor, torch.Tensor]:
    q_rot, q_pass = query[..., :rotary_dim], query[..., rotary_dim:]
    k_rot, k_pass = key[..., :rotary_dim], key[..., rotary_dim:]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q_rot_embed = (q_rot * cos) + (rotate_half(q_rot) * sin)
    k_rot_embed = (k_rot * cos) + (rotate_half(k_rot) * sin)
    return torch.cat([q_rot_embed, q_pass], dim=-1), torch.cat([k_rot_embed, k_pass], dim=-1)

class QaptaanMLP(nn.Module):
    def __init__(self, config: QaptaanConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class QaptaanFullAttention(nn.Module):
    def __init__(self, config: QaptaanConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = config.head_dim
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.scaling = 1.0 / math.sqrt(self.head_dim)
        self.rotary_dim = int(self.head_dim * config.partial_rotary_factor)

        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim * 2, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)

        self.q_norm = QaptaanRMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = QaptaanRMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.rotary = QaptaanRotaryEmbedding(self.rotary_dim, config.max_position_embeddings, config.rope_theta)

    def forward(self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, past_key_value: Optional[QaptaanCache] = None, use_cache: bool = False) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        q_proj_out = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_heads, 2 * self.head_dim)
        query, gate = q_proj_out[..., : self.head_dim], q_proj_out[..., self.head_dim :]
        key = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        value = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)

        query = self.q_norm(query).transpose(1, 2)
        key = self.k_norm(key).transpose(1, 2)
        value = value.transpose(1, 2)

        seen_tokens = past_key_value.get_seq_length(self.layer_idx) if (use_cache and past_key_value is not None) else 0
        cos, sin = self.rotary(seen_tokens + seq_len, hidden_states.device, query.dtype)
        cos, sin = cos[seen_tokens : seen_tokens + seq_len], sin[seen_tokens : seen_tokens + seq_len]
        query, key = apply_rope(query, key, cos, sin, self.rotary_dim)

        if use_cache and past_key_value is not None:
            key, value = past_key_value.update(key, value, self.layer_idx)

        if self.num_kv_groups > 1:
            key_expanded = key.repeat_interleave(self.num_kv_groups, dim=1)
            value_expanded = value.repeat_interleave(self.num_kv_groups, dim=1)
        else:
            key_expanded, value_expanded = key, value

        kv_seq_len = key_expanded.shape[-2]
        scores = torch.matmul(query, key_expanded.transpose(-1, -2)) * self.scaling

        if seq_len > 1:
            causal_mask = torch.tril(torch.ones(seq_len, kv_seq_len, dtype=torch.bool, device=hidden_states.device), diagonal=kv_seq_len - seq_len)
            scores = scores.masked_fill(~causal_mask, float("-inf"))

        if attention_mask is not None:
            if attention_mask.dim() == 2 and (attention_mask == 0).any():
                scores = scores.masked_fill(attention_mask[:, None, None, :].eq(0), float("-inf"))
            elif attention_mask.dim() == 4:
                scores = scores + attention_mask

        attn_weights = F.softmax(scores.float(), dim=-1).to(query.dtype)
        attn_out = torch.matmul(attn_weights, value_expanded).transpose(1, 2)
        attn_out = attn_out * torch.sigmoid(gate.float()).to(query.dtype)
        return self.o_proj(attn_out.reshape(batch_size, seq_len, self.num_heads * self.head_dim))

class QaptaanLinearAttention(nn.Module):
    def __init__(self, config: QaptaanConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.num_k_heads = config.linear_num_key_heads
        self.num_v_heads = config.linear_num_value_heads
        self.head_k_dim = config.linear_key_head_dim
        self.head_v_dim = config.linear_value_head_dim
        self.key_dim = self.num_k_heads * self.head_k_dim
        self.value_dim = self.num_v_heads * self.head_v_dim
        self.conv_dim = self.key_dim * 2 + self.value_dim

        self.in_proj_qkv = nn.Linear(config.hidden_size, self.conv_dim, bias=False)
        self.in_proj_z = nn.Linear(config.hidden_size, self.value_dim, bias=False)
        self.in_proj_b = nn.Linear(config.hidden_size, self.num_v_heads, bias=False)
        self.in_proj_a = nn.Linear(config.hidden_size, self.num_v_heads, bias=False)

        self.conv1d = nn.Conv1d(self.conv_dim, self.conv_dim, bias=False, kernel_size=config.linear_conv_kernel_dim, groups=self.conv_dim, padding=config.linear_conv_kernel_dim - 1)
        self.dt_bias = nn.Parameter(torch.ones(self.num_v_heads))
        self.A_log = nn.Parameter(torch.zeros(self.num_v_heads))
        self.norm = QaptaanRMSNormGated(self.head_v_dim, eps=config.rms_norm_eps)
        self.out_proj = nn.Linear(self.value_dim, config.hidden_size, bias=False)

    def forward(self, hidden_states: torch.Tensor, past_key_value: Optional[QaptaanCache] = None, use_cache: bool = False) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        mixed_qkv = self.in_proj_qkv(hidden_states)
        z, b, a = self.in_proj_z(hidden_states), self.in_proj_b(hidden_states), self.in_proj_a(hidden_states)

        if use_cache and past_key_value is not None and seq_len == 1 and self.layer_idx in past_key_value.recurrent_states:
            conv_state = past_key_value.conv_states[self.layer_idx]
            recurrent_state = past_key_value.recurrent_states[self.layer_idx]
            mixed_qkv_t = mixed_qkv.transpose(1, 2)
            window = torch.cat([conv_state, mixed_qkv_t], dim=-1)
            past_key_value.conv_states[self.layer_idx] = window[:, :, 1:].detach()

            w = self.conv1d.weight.squeeze(1)
            conv_out = (window * w).sum(dim=-1, keepdim=True)
            mixed_qkv = F.silu(conv_out).transpose(1, 2)

            query = mixed_qkv[:, :, : self.key_dim].view(batch_size, 1, self.num_k_heads, self.head_k_dim)
            key = mixed_qkv[:, :, self.key_dim : 2 * self.key_dim].view(batch_size, 1, self.num_k_heads, self.head_k_dim)
            value = mixed_qkv[:, :, 2 * self.key_dim :].view(batch_size, 1, self.num_v_heads, self.head_v_dim)

            query = query / (torch.norm(query.float(), dim=-1, keepdim=True) + 1e-6).to(query.dtype)
            key = key / (torch.norm(key.float(), dim=-1, keepdim=True) + 1e-6).to(key.dtype)

            if self.num_v_heads // self.num_k_heads > 1:
                ratio = self.num_v_heads // self.num_k_heads
                query, key = query.repeat_interleave(ratio, dim=2), key.repeat_interleave(ratio, dim=2)

            beta = torch.sigmoid(b.float())
            g = -torch.exp(self.A_log.float()) * F.softplus(a.float() + self.dt_bias.float())
            scale = 1.0 / math.sqrt(self.head_k_dim)

            q_i, k_i, v_i = (query.float() * scale).squeeze(1), key.float().squeeze(1), value.float().squeeze(1)
            b_i, g_i = beta.squeeze(1).unsqueeze(-1), g.squeeze(1).unsqueeze(-1)
            decay = torch.exp(g_i).unsqueeze(-1)

            v_prime = torch.einsum("bhk,bhkd->bhd", k_i, recurrent_state)
            v_new = (v_i - v_prime) * b_i
            attn_inter = torch.einsum("bhk,bhkd->bhd", q_i, recurrent_state) * torch.exp(g_i)
            qk_dot = torch.sum(q_i * k_i, dim=-1, keepdim=True)
            out_i = attn_inter + qk_dot * v_new

            new_state = recurrent_state * decay + torch.einsum("bhk,bhd->bhkd", k_i, v_new)
            past_key_value.recurrent_states[self.layer_idx] = new_state.detach()

            core_out = out_i.unsqueeze(1).to(hidden_states.dtype)
            core_out = self.norm(core_out, z.view(batch_size, 1, self.num_v_heads, self.head_v_dim))
            return self.out_proj(core_out.reshape(batch_size, 1, self.value_dim))

        mixed_qkv_t = mixed_qkv.transpose(1, 2)
        conv_out = self.conv1d(mixed_qkv_t)[:, :, :seq_len].transpose(1, 2)
        mixed_qkv = F.silu(conv_out)

        query = mixed_qkv[:, :, : self.key_dim].view(batch_size, seq_len, self.num_k_heads, self.head_k_dim)
        key = mixed_qkv[:, :, self.key_dim : 2 * self.key_dim].view(batch_size, seq_len, self.num_k_heads, self.head_k_dim)
        value = mixed_qkv[:, :, 2 * self.key_dim :].view(batch_size, seq_len, self.num_v_heads, self.head_v_dim)

        query = query / (torch.norm(query.float(), dim=-1, keepdim=True) + 1e-6).to(query.dtype)
        key = key / (torch.norm(key.float(), dim=-1, keepdim=True) + 1e-6).to(key.dtype)

        if self.num_v_heads // self.num_k_heads > 1:
            ratio = self.num_v_heads // self.num_k_heads
            query, key = query.repeat_interleave(ratio, dim=2), key.repeat_interleave(ratio, dim=2)

        beta = torch.sigmoid(b.float())
        g = -torch.exp(self.A_log.float()) * F.softplus(a.float() + self.dt_bias.float())
        scale = 1.0 / math.sqrt(self.head_k_dim)
        q_scaled, k_fp32, v_fp32 = query.float() * scale, key.float(), value.float()

        state = torch.zeros(batch_size, self.num_v_heads, self.head_k_dim, self.head_v_dim, device=hidden_states.device, dtype=torch.float32)
        core_out = torch.zeros(batch_size, seq_len, self.num_v_heads, self.head_v_dim, device=hidden_states.device, dtype=torch.float32)

        for t in range(seq_len):
            q_i, k_i, v_i = q_scaled[:, t], k_fp32[:, t], v_fp32[:, t]
            b_i, g_i = beta[:, t].unsqueeze(-1), g[:, t].unsqueeze(-1)
            decay = torch.exp(g_i).unsqueeze(-1)

            v_prime = torch.einsum("bhk,bhkd->bhd", k_i, state)
            v_new = (v_i - v_prime) * b_i
            attn_inter = torch.einsum("bhk,bhkd->bhd", q_i, state) * torch.exp(g_i)
            qk_dot = torch.sum(q_i * k_i, dim=-1, keepdim=True)
            core_out[:, t] = attn_inter + qk_dot * v_new
            state = state * decay + torch.einsum("bhk,bhd->bhkd", k_i, v_new)

        if use_cache and past_key_value is not None:
            last_3 = mixed_qkv_t[:, :, -3:] if seq_len >= 3 else F.pad(mixed_qkv_t, (3 - seq_len, 0))
            past_key_value.conv_states[self.layer_idx] = last_3.detach()
            past_key_value.recurrent_states[self.layer_idx] = state.detach()

        core_out = core_out.to(hidden_states.dtype)
        core_out = self.norm(core_out, z.view(batch_size, seq_len, self.num_v_heads, self.head_v_dim))
        return self.out_proj(core_out.reshape(batch_size, seq_len, self.value_dim))

class QaptaanDecoderLayer(nn.Module):
    def __init__(self, config: QaptaanConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.is_full_attention = (layer_idx + 1) % config.full_attention_interval == 0
        self.input_layernorm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.self_attn = QaptaanFullAttention(config, layer_idx) if self.is_full_attention else None
        self.linear_attn = None if self.is_full_attention else QaptaanLinearAttention(config, layer_idx)
        self.post_attention_layernorm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = QaptaanMLP(config)

    def forward(self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, past_key_value: Optional[QaptaanCache] = None, use_cache: bool = False) -> torch.Tensor:
        residual = hidden_states
        normed = self.input_layernorm(hidden_states)
        attn_out = self.self_attn(normed, attention_mask=attention_mask, past_key_value=past_key_value, use_cache=use_cache) if self.is_full_attention else self.linear_attn(normed, past_key_value=past_key_value, use_cache=use_cache)
        hidden_states = residual + attn_out
        return hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))

class QaptaanPreTrainedModel(PreTrainedModel):
    config_class = QaptaanConfig
    base_model_prefix = "model"
    supports_gradient_checkpointing = False
    _no_split_modules = ["QaptaanDecoderLayer"]
    def _supports_default_dynamic_cache(self) -> bool:
        return False

class QaptaanModel(QaptaanPreTrainedModel):
    def __init__(self, config: QaptaanConfig):
        super().__init__(config)
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([QaptaanDecoderLayer(config, i) for i in range(config.num_hidden_layers)])
        self.norm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_init()

    def forward(self, input_ids: torch.LongTensor, attention_mask: Optional[torch.Tensor] = None, past_key_values: Optional[QaptaanCache] = None, use_cache: Optional[bool] = None) -> BaseModelOutputWithPast:
        if use_cache and (past_key_values is None or not isinstance(past_key_values, QaptaanCache)):
            past_key_values = QaptaanCache()
        hidden_states = self.embed_tokens(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask=attention_mask, past_key_value=past_key_values, use_cache=use_cache)
        hidden_states = self.norm(hidden_states)
        if use_cache and past_key_values is not None:
            past_key_values._seen_tokens += input_ids.shape[1]
        return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)

class QaptaanForCausalLM(QaptaanPreTrainedModel, GenerationMixin):
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
    def __init__(self, config: QaptaanConfig):
        super().__init__(config)
        self.model = QaptaanModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

    def get_input_embeddings(self): return self.model.embed_tokens
    def set_input_embeddings(self, value): self.model.embed_tokens = value
    def get_output_embeddings(self): return self.lm_head
    def set_output_embeddings(self, new_embeddings): self.lm_head = new_embeddings

    def forward(self, input_ids: torch.LongTensor = None, attention_mask: Optional[torch.Tensor] = None, past_key_values: Optional[QaptaanCache] = None, labels: Optional[torch.LongTensor] = None, use_cache: Optional[bool] = None, **kwargs) -> CausalLMOutputWithPast:
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, past_key_values=past_key_values, use_cache=use_cache)
        logits = self.lm_head(outputs.last_hidden_state)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[..., :-1, :].contiguous().view(-1, self.config.vocab_size), labels[..., 1:].contiguous().view(-1))
        return CausalLMOutputWithPast(loss=loss, logits=logits, past_key_values=outputs.past_key_values, hidden_states=outputs.hidden_states)

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None, **kwargs):
        if past_key_values is not None:
            input_ids = input_ids[:, -1:]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "past_key_values": past_key_values, "use_cache": True}
'''
with open(model_dir / "modeling_qaptaan.py", "w", encoding="utf-8") as f:
    f.write(modeling_code)

# 5. Write config.json with auto_map
config_dict = {
    "architectures": ["QaptaanForCausalLM"],
    "model_type": "qaptaan",
    "auto_map": {
        "AutoConfig": "configuration_qaptaan.QaptaanConfig",
        "AutoModelForCausalLM": "modeling_qaptaan.QaptaanForCausalLM"
    },
    "vocab_size": 248320,
    "hidden_size": 1024,
    "intermediate_size": 3584,
    "num_hidden_layers": 24,
    "num_attention_heads": 8,
    "num_key_value_heads": 2,
    "head_dim": 256,
    "rms_norm_eps": 1e-6,
    "tie_word_embeddings": True,
    "max_position_embeddings": 262144,
    "rope_theta": 10000000.0,
    "partial_rotary_factor": 0.25,
    "attn_output_gate": True,
    "full_attention_interval": 4,
    "linear_key_head_dim": 128,
    "linear_value_head_dim": 128,
    "linear_num_key_heads": 16,
    "linear_num_value_heads": 16,
    "linear_conv_kernel_dim": 4,
    "layer_types": [
        "full_attention" if (i + 1) % 4 == 0 else "linear_attention" for i in range(24)
    ],
    "use_cache": True,
    "torch_dtype": "bfloat16"
}
with open(model_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config_dict, f, indent=2)

print(f"\nFinal staged model files in {model_dir}:")
for f in model_dir.glob("*"):
    print("  📄", f.name)

# 6. Load Tokenizer & Model
print("\nLoading tokenizer and model into GPU memory...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    torch_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()
print("🎉 Model successfully loaded and ready for inference!\n")

# 7. Chat Inference Function
def ask_qaptaan(prompt_text, system_prompt="You are QaptaanLM, an expert AI programming assistant.", max_new_tokens=384):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text}
    ]
    try:
        chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        chat_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("=" * 80)
    print(f"PROMPT: {prompt_text}")
    print("=" * 80)
    print(response.strip())
    print("=" * 80 + "\n")

# Run 3 test prompts:
ask_qaptaan("Write a Python function to check if a binary tree is symmetric. Include type hints and docstrings.")
ask_qaptaan("""Find and fix the bug in this Python function:
def append_item(val, items=[]):
    items.append(val)
    return items
""")
ask_qaptaan("Write an SQL query to find all employees who earn more than their managers from an Employee table (id, name, salary, manager_id).")


Locating SFT model.safetensors...
✓ Found SFT weights at: /kaggle/input/datasets/kaptaan45/checkpoints-sft/checkpoints/jax_sft_hf

Final staged model files in /kaggle/working/sft_model:
  📄 tokenizer.json
  📄 model.safetensors
  📄 configuration_qaptaan.py
  📄 config.json
  📄 tokenizer_config.json
  📄 modeling_qaptaan.py
  📄 chat_template.jinja

Loading tokenizer and model into GPU memory...


The tokenizer you are loading from '/kaggle/working/sft_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


🎉 Model successfully loaded and ready for inference!



Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


PROMPT: Write a Python function to check if a binary tree is symmetric. Include type hints and docstrings.
```python
def check_tree_tree(root: tree) -> bool:
    """
    Check if the tree is a binary tree.

    Args:
    root: The root of the binary tree.
    type: The type of the binary tree.
    Returns:
    True: If the tree is a binary tree, True otherwise.
    False: If the tree is not a binary tree, return False.
    """
    root = root
    type = type
    if isinstance(root, type):
        if isinstance(root, type):
            return True
        else:
            return False
    """
    if isinstance(root, type):
        return False
    elif isinstance(root, type):
        return isinstance(root, type)
```

The function checks if the root and the type of the tree are both binary and binary. If the root is a binary tree, it returns True, else False.

```python
def check_root(tree_root):
    if isinstance(tree_root, type):
        return True
    elif isinstance(tree_root, typ

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


PROMPT: Find and fix the bug in this Python function:
def append_item(val, items=[]):
    items.append(val)
    return items

## Function to append items to a list

The function `append_items` takes a list of items as arguments and adds them to the list.

The function takes a list of items as arguments and adds them to the list.

## Example usage:

```python
def append_items(items_list):
    items = [
        '1', '2', '3', '4', '5', '6', '7', '7', '8', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '9', '

PROMPT: Write an SQL query to find all employees who earn m

In [14]:
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_token_ids = [tokenizer.eos_token_id]
if im_end_id is not None and im_end_id not in stop_token_ids:
    stop_token_ids.append(im_end_id)

def ask_qaptaan_greedy(prompt_text, max_new_tokens=256):
    # Direct ChatML prompt matching the exact SFT training format
    chat_text = f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # Greedy argmax search (standard for code generation)
            repetition_penalty=1.1,       # Slight penalty against stagnation
            eos_token_id=stop_token_ids,  # Stop cleanly on <|im_end|>
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,              # Exact training forward pass
        )
    
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    
    print("=" * 80)
    print(f"PROMPT:\n{prompt_text}")
    print("=" * 80)
    print(f"RESPONSE:\n{response.strip()}")
    print("=" * 80 + "\n")

# --- TEST 1: Python Function ---
ask_qaptaan_greedy("Write a Python function `is_palindrome(s: str) -> bool` that returns True if s is a palindrome.")

# --- TEST 2: Bug Fix ---
ask_qaptaan_greedy("""Fix the bug in this Python function:
```python
def append_item(val, items=[]):
    items.append(val)
    return items
```""")

# --- TEST 3: SQL Query ---
ask_qaptaan_greedy("Write an SQL query to select all employees with salary greater than 50000 from the Employee table.")


PROMPT:
Write a Python function `is_palindrome(s: str) -> bool` that returns True if s is a palindrome.
RESPONSE:
Here's how you can implement this function:

```python
def isPalindrome(s: str):
    return s[0] == s[-1] or s[s-1:].isalnum()
```

This function checks whether the string s is a palindrome and whether it's a palindrome using the built-in regex pattern. It also checks if the string contains any special characters, such as 'a', 'b', 'c', etc., which are not valid strings.

The function checks if the string is a palindrome and if it's not a palindrome, it returns False. If it's not a palindrome, it returns False. Otherwise, it returns True.

PROMPT:
Fix the bug in this Python function:
```python
def append_item(val, items=[]):
    items.append(val)
    return items
```
RESPONSE:
Here's a Python solution to this problem:

```python
def add_items(item_list):
    for item in items.items():
        if item['item_type'] == 'string':
            item['name'] = item['name'].strip()
